In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

In [ ]:
%pip install -q bitsandbytes accelerate
%pip install groq

In [ ]:
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

# 1. Setup Environment & Baseline RAG
Thiết lập môi trường và tạo baseline RAG system

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install requirements
#%pip install -r ../requirements.txt

In [ ]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"

In [ ]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
# Ensure faiss-cpu is installed for HybridRetriever
%pip install -q "faiss-cpu"
print("✅ faiss-cpu installation command executed.")

In [ ]:
from src.config.config import config
from src.data.data_loader import DataLoader
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.llm_generator import LLMGenerator

print("✅ Imports successful")

## Load Data

In [ ]:
data_loader = DataLoader("../data")

# Check available datasets
datasets = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]

for ds in datasets:
    stats = data_loader.get_dataset_stats(ds)
    print(f"{ds}: {stats}")

## Setup Retriever

In [ ]:
retriever = HybridRetriever(
    vector_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    bm25_weight=config.retriever.hybrid_bm25_weight,
    vector_weight=config.retriever.hybrid_vector_weight
)

all_documents = []

print("Loading VHealthQA...")
vhealth_train = data_loader.load_dataset("vhealthqa", "train")
for sample in vhealth_train:
    text = sample.get("ground_truth") or sample.get("answer") or ""
    if text and not text.startswith("http") and len(text) > 50:
        all_documents.append(text.strip())
print(f"VHealthQA: {len(all_documents)} docs")

print("Loading UIT-ViQuAD2...")
viquad_train = data_loader.load_dataset("uit_viquad2", "train")
start_count = len(all_documents)
for sample in viquad_train:
    text = sample.get("context") or sample.get("paragraph") or ""
    if text and len(text) > 50:
        all_documents.append(text.strip())
print(f"UIT-ViQuAD2: +{len(all_documents) - start_count} docs")

print("Loading Vietnamese_RAG...")
try:
    vnrag_train = data_loader.load_dataset("vietnamese_rag", "train")
    start_count = len(all_documents)
    for sample in vnrag_train:
        text = (
            sample.get("context") or
            sample.get("passage") or
            sample.get("text") or
            sample.get("document") or
            ""
        )
        if text and not text.startswith("http") and len(text) > 50:
            all_documents.append(text.strip())
    print(f"Vietnamese_RAG: +{len(all_documents) - start_count} docs")
except Exception as e:
    print(f"Vietnamese_RAG failed: {e}")

print("Loading VIMPA...")
try:
    vimqa_train = data_loader.load_dataset("vimpa", "train")
    start_count = len(all_documents)
    for sample in vimqa_train:
        contexts = sample.get("context") or sample.get("contexts") or []

        if isinstance(contexts, list):
            for ctx in contexts:
                if isinstance(ctx, dict):
                    text = ctx.get("text") or ctx.get("content") or ""
                else:
                    text = str(ctx)
                if text and len(text) > 50:
                    all_documents.append(text.strip())
        elif isinstance(contexts, str) and len(contexts) > 50:
            all_documents.append(contexts.strip())
    print(f"VIMPA: +{len(all_documents) - start_count} docs")
except Exception as e:
    print(f"VIMPA failed: {e}")

print(f"\nProcessing corpus...")
seen = set()
unique_docs = []
for doc in all_documents:
    doc_clean = doc.strip()
    if (doc_clean not in seen and 50 < len(doc_clean) <= 1500 and not doc_clean.startswith("http")):
        seen.add(doc_clean)
        unique_docs.append(doc_clean)

del all_documents

print(f"\nCorpus Statistics:")
print(f"  - Total unique docs: {len(unique_docs)}")
print(f"  - Avg length: {sum(len(d) for d in unique_docs) / len(unique_docs):.0f} chars")
print(f"  - Min length: {min(len(d) for d in unique_docs)}")
print(f"  - Max length: {max(len(d) for d in unique_docs)}")
print(f"\nSample doc ({len(unique_docs[0])} chars):")
print(f"{unique_docs[0][:200]}...")

retriever.build_index(unique_docs)
print(f"Index built with {len(unique_docs)} documents")

del unique_docs

## Setup LLM

In [ ]:
# Initialize LLM
#llm = LLMGenerator(provider="huggingface", hf_model_name="Qwen/Qwen2.5-3B-Instruct")
llm = LLMGenerator(
    provider="groq",
    groq_api_key=GROQ_API_KEY,
    groq_model="llama-3.1-8b-instant"
)

# Test generation
test_prompt = "Xin chào, hôm nay bạn khỏe không?"
response = llm.generate(test_prompt, [], max_tokens=100)
print(f"Response: {response}")

## Baseline RAG Pipeline

In [ ]:
def baseline_rag(query, retriever, llm):
    """Simple baseline RAG without query rewriting"""
    # Retrieve
    docs = retriever.retrieve([query], top_k=5)

    print(docs)

    # Generate
    answer = llm.generate(query, docs, max_tokens=100)

    return answer, docs

# Test
test_query = "Bệnh đái tháo đường có triệu chứng là gì?"
answer, docs = baseline_rag(test_query, retriever, llm)

print(f"Query: {test_query}\n")
print(f"Answer: {answer}\n")
print(f"Retrieved {len(docs)} documents")

## Evaluation

In [ ]:
from evaluation.evaluator import RAGEvaluator

evaluator = RAGEvaluator()

# Test case 1: Perfect match
pred = "Bệnh đái tháo đường là tình trạng lượng glucose cao"
ground = "Bệnh đái tháo đường là tình trạng lượng glucose cao"

metrics = evaluator.calculate_all(
    query="Bệnh đái tháo đường là gì?",
    prediction=pred,
    context=[{"text": "Bệnh đái tháo đường...", "score": 0.9}],
    ground_truth=ground
)
print(f"Test 1 (perfect match): {metrics}")
# Expected: em=1.0, f1~1.0, similarity~1.0

# Test case 2: Partial overlap
pred = "Bệnh đái tháo đường là tình trạng glucose"
ground = "Bệnh đái tháo đường là tình trạng lượng glucose cao"

metrics = evaluator.calculate_all(
    query="Bệnh đái tháo đường là gì?",
    prediction=pred,
    context=[{"text": "Bệnh đái tháo đường là tình trạng lượng glucose cao", "score": 0.9}],
    ground_truth=ground
)
print(f"Test 2 (partial): {metrics}")
# Expected: f1 > 0, similarity > 0